In [1]:
import sys
sys.path.append('../src')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from dataset import prepare_data
from model   import RISFaultDetector, print_model_summary
from utils   import get_device

device = get_device()

[utils] Using device: cpu


In [2]:
train_loader, val_loader, test_loader, scaler, pos_weight = prepare_data(
    batch_size=256
)
print(f'pos_weight = {pos_weight:.3f}')
print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')

# Peek at one batch
X_batch, y_batch = next(iter(train_loader))
print(f'X batch shape: {X_batch.shape}  (batch_size x 45 features)')
print(f'y batch shape: {y_batch.shape}  (batch_size x 100 pixel labels)')

[dataset] Loading preprocessed data from disk...
[dataset] pos_weight = 2.916  (handles 25.5% failure rate imbalance)
[dataset] DataLoaders ready — batch size: 256
pos_weight = 2.916
Train batches: 137
Val   batches: 30
X batch shape: torch.Size([256, 45])  (batch_size x 45 features)
y batch shape: torch.Size([256, 100])  (batch_size x 100 pixel labels)


In [3]:
model = RISFaultDetector(n_features=45, n_pixels=100).to(device)
print_model_summary(model)

# Loss: BCE with pos_weight for class imbalance
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

# Optimizer: Adam lr=0.1 (paper Section V-A)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

print('Model ready!')


  RISFaultDetector — Architecture Summary
  network.0.weight               [32, 45]             1,440 params
  network.0.bias                 [32]                 32 params
  network.3.weight               [64, 32]             2,048 params
  network.3.bias                 [64]                 64 params
  network.6.weight               [128, 64]            8,192 params
  network.6.bias                 [128]                128 params
  network.9.weight               [100, 128]           12,800 params
  network.9.bias                 [100]                100 params
--------------------------------------------------
  Total trainable parameters: 24,804

Model ready!


In [4]:
from sklearn.metrics import f1_score

def run_epoch(model, loader, optimizer, criterion, device, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            logits = model(X)
            loss   = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(X)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_preds.append((probs >= 0.5).astype(int))
            all_labels.append(y.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels.flatten(), all_preds.flatten(), zero_division=0)
    return avg_loss, f1

print('Running 5-epoch sanity check...')
for ep in range(1, 6):
    tl, tf = run_epoch(model, train_loader, optimizer, criterion, device, train=True)
    vl, vf = run_epoch(model, val_loader,   optimizer, criterion, device, train=False)
    print(f'Epoch {ep} | Train Loss: {tl:.4f} F1: {tf:.4f} | Val Loss: {vl:.4f} F1: {vf:.4f}')

Running 5-epoch sanity check...
Epoch 1 | Train Loss: 1.0368 F1: 0.3393 | Val Loss: 1.0304 F1: 0.3337
Epoch 2 | Train Loss: 1.0336 F1: 0.3368 | Val Loss: 1.0306 F1: 0.3125
Epoch 3 | Train Loss: 1.0335 F1: 0.3397 | Val Loss: 1.0309 F1: 0.2866
Epoch 4 | Train Loss: 1.0334 F1: 0.3377 | Val Loss: 1.0306 F1: 0.3157
Epoch 5 | Train Loss: 1.0335 F1: 0.3364 | Val Loss: 1.0311 F1: 0.3590


In [ ]:
# Reset model and optimizer for clean training
model     = RISFaultDetector(n_features=45, n_pixels=100).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=20)

EPOCHS = 500
train_losses, val_losses, train_f1s, val_f1s = [], [], [], []
best_val_f1, best_state = 0.0, None

for ep in range(1, EPOCHS + 1):
    tl, tf = run_epoch(model, train_loader, optimizer, criterion, device, True)
    vl, vf = run_epoch(model, val_loader,   optimizer, criterion, device, False)
    train_losses.append(tl); val_losses.append(vl)
    train_f1s.append(tf);    val_f1s.append(vf)
    scheduler.step(vf)

    if vf > best_val_f1:
        best_val_f1 = vf
        best_state  = {k: v.clone() for k, v in model.state_dict().items()}

    if ep % 50 == 0 or ep == 1:
        print(f'Ep {ep:>4}/{EPOCHS} | '
              f'T_loss={tl:.4f} T_F1={tf:.4f} | '
              f'V_loss={vl:.4f} V_F1={vf:.4f}')

print(f'\nBest Val F1: {best_val_f1:.4f}')

# Save best model
import os
os.makedirs('../outputs/models', exist_ok=True)
torch.save(best_state, '../outputs/models/best_model.pth')
print('Best model saved → outputs/models/best_model.pth')

Ep    1/500 | T_loss=1.0436 T_F1=0.3424 | V_loss=1.0366 V_F1=0.3397
Ep   50/500 | T_loss=1.0317 T_F1=0.3445 | V_loss=1.0302 V_F1=0.3322
Ep  100/500 | T_loss=1.0315 T_F1=0.3465 | V_loss=1.0302 V_F1=0.3316
Ep  150/500 | T_loss=1.0314 T_F1=0.3468 | V_loss=1.0302 V_F1=0.3317
Ep  200/500 | T_loss=1.0314 T_F1=0.3469 | V_loss=1.0302 V_F1=0.3317
Ep  250/500 | T_loss=1.0314 T_F1=0.3469 | V_loss=1.0302 V_F1=0.3317
Ep  300/500 | T_loss=1.0314 T_F1=0.3469 | V_loss=1.0302 V_F1=0.3317
Ep  350/500 | T_loss=1.0314 T_F1=0.3469 | V_loss=1.0302 V_F1=0.3317
Ep  400/500 | T_loss=1.0314 T_F1=0.3470 | V_loss=1.0302 V_F1=0.3317
Ep  450/500 | T_loss=1.0314 T_F1=0.3470 | V_loss=1.0302 V_F1=0.3317
Ep  500/500 | T_loss=1.0314 T_F1=0.3470 | V_loss=1.0302 V_F1=0.3318

Best Val F1: 0.3397
Best model saved → outputs/models/best_model.pth
